# LLM 2일차 (1) — LangChain 체인 : 프롬프트 | LLM | 출력 파서 · 고객 리뷰 감성 분석 체인

수업 필기 `LangChain_실습1` · `LangChain_실습2` 를 정리 · 개선한 노트입니다.
(교재 `AI Agent` 교재10 **p.18~24** — 23. LangChain · 정의 · 라이브러리 vs 프레임워크 · 체인 · LCEL · Chain 생성 및 실행 실습 · 고객 리뷰 감성 분석 애플리케이션)

```python
chain = prompt | llm | output_parser       # LCEL : '|' 로 이어 붙인다 — 앞의 출력이 뒤의 입력
chain.invoke({'topic': '양자 컴퓨터'})       # 딕셔너리로 프롬프트 변수 값을 넣는다
```

- 챗봇(필기 `LangChain_실습3`)은 [`langchain_chatbot_memory.ipynb`](./langchain_chatbot_memory.ipynb), 도구 · 에이전트(필기 `LangChain_실습4` · `실습5`)는 [`agent_tools_tavily.ipynb`](./agent_tools_tavily.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 과 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | API 대신 **정해 둔 답을 내는 가짜 LLM** 으로 체인 · 에이전트의 **흐름**만 실제로 돌린 셀 (답 내용은 의미 없음) |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (프롬프트 채우기 · 토큰 세기 · 도구 정보 · 기록 분석) |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langchain` · `langchain-core` | `ChatPromptTemplate` · `StrOutputParser` · LCEL `\|` | `pip install langchain` |
| `langchain-openai` | `ChatOpenAI` (OpenAI 모델 연결) | `pip install langchain-openai` |
| `python-dotenv` | `.env` 의 API 키 읽기 | `pip install python-dotenv` |
| `pydantic` | 감성 분석 결과 형식(구조화 출력) | langchain 과 함께 설치됨 |

**API 키** : 노트북 폴더(또는 상위 폴더)에 `.env` → `OPENAI_API_KEY="sk-…"` (**GitHub 에 올리지 않는다**). 수업 버전 : `langchain 1.5.0` · `langchain-core 1.8.0` · `langchain-openai 1.7.4`.
**실행 순서** : 맨 위부터 **모두 실행**. `# ⏳` 셀이 실제 OpenAI 호출(인터넷 · 요금, `gpt-5-nano` 라 한 번에 1센트 미만).

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langchain langchain-openai python-dotenv

In [2]:
# 필요한 함수 / 클래스 임포트
from langchain_openai import ChatOpenAI                   # LLM : OpenAI 모델과 연결
from langchain_core.prompts import ChatPromptTemplate     # 프롬프트 틀
from langchain_core.output_parsers import StrOutputParser # 출력 파서 : 응답 객체 → 글자(str)

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
import os
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))      # 지금 폴더에서 위로 올라가며 .env 를 찾는다
print('.env 읽음 :', loaded)
print('OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')), '| TAVILY_API_KEY 있음 :', bool(os.getenv('TAVILY_API_KEY')))

.env 읽음 : False
OPENAI_API_KEY 있음 : False | TAVILY_API_KEY 있음 : False


## 1. 첫 번째 구성 요소 — 프롬프트 (교재 p.21~22)

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 친절하고 정확한 답변을 하는 AI 비서야. 사용자의 질문에 100 토큰 이내로 간결하게 답변해야 해'),
    ('user', '{topic}에 대해서 설명해 줘'),                 # {topic} : 체인을 실행할 때 채울 빈칸
])
print('채울 변수 :', prompt.input_variables)

# 빈칸을 채우면 실제로 LLM 에 보낼 메시지 목록이 된다 — API 없이 확인 가능
for m in prompt.format_messages(topic='양자 컴퓨터'):
    print(f'{type(m).__name__:14s} {m.content}')

채울 변수 : ['topic']
SystemMessage  너는 친절하고 정확한 답변을 하는 AI 비서야. 사용자의 질문에 100 토큰 이내로 간결하게 답변해야 해
HumanMessage   양자 컴퓨터에 대해서 설명해 줘


`print(prompt)` 는 내부 구조(`input_variables=['topic'] … SystemMessagePromptTemplate(…)`)를 길게 보여 준다. 실제로 무엇이 전송되는지는 **`format_messages()`** 로 채워 보면 바로 보인다.

## 2. 두 번째 · 세 번째 구성 요소 — LLM · 출력 파서

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

llm = ChatOpenAI(model='gpt-5-nano')
print(llm)                                     # 키는 SecretStr('**********') 로 가려져 출력된다

metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-openai': '1.7.4'}} profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'mid_conversation_system_messages': True, 'mid_conversation_tools': False} client=<openai.resources.chat.completions.completions.Completions object at 0x000002C2B00D8530> async_client=<openai.resources.chat.co

수업 출력의 `profile` 에 모델 정보가 들어 있다 — `max_input_tokens 272000` · `max_output_tokens 128000` · `temperature False`(gpt-5 계열은 temperature 조절 불가) · `reasoning_effort_levels ['minimal', 'low', 'medium', 'high']` · `tool_calling True` · `structured_output True`(§4 에서 사용).

In [6]:
output_parser = StrOutputParser()
print(repr(output_parser))                     # print(output_parser) 는 빈 줄만 나온다 (원본 cell 7)

StrOutputParser()


## 3. 체인 만들기 · 실행 (교재 p.21~23)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

chain = prompt | llm | output_parser           # LCEL : 프롬프트 → LLM → 출력 파서
output = chain.invoke({'topic': '양자 컴퓨터'})
print(f'chain 실행의 결과: \n{output}')

chain 실행의 결과: 
양자 컴퓨터는 정보를 0/1 두 상태로만 다루는 대신 큐비트(qubit)로 처리합니다. 큐비트는 중첩으로 한 번에 여러 상태를 가질 수 있고, 얽힘으로 큐비트 간 강한 상호작용이 가능해 병렬 계산이 이뤄집니다. 양자 게이트로 연산하며, 특정 문제에서 고전 컴퓨터보다 속도 이점을 낼 수 있습니다. 대표 알고리즘으로 쇼어(소인수 분해)와 그로버(탐색)가 있습니다. 현재는 노이즈로 규모가 작고 연구 중(NISQ). 응용 분야로는 화학 시뮬레이션, 최적화, 암호학 등이 있습니다.


In [8]:
# 수업 결과(📼)의 길이 — '100 토큰 이내' 를 지켰을까 (tiktoken 으로 직접 세기)
import tiktoken
enc = tiktoken.get_encoding('o200k_base')              # gpt-4o · gpt-5 계열 토크나이저
recorded_answer = '''양자 컴퓨터는 정보를 0/1 두 상태로만 다루는 대신 큐비트(qubit)로 처리합니다. 큐비트는 중첩으로 한 번에 여러 상태를 가질 수 있고, 얽힘으로 큐비트 간 강한 상호작용이 가능해 병렬 계산이 이뤄집니다. 양자 게이트로 연산하며, 특정 문제에서 고전 컴퓨터보다 속도 이점을 낼 수 있습니다. 대표 알고리즘으로 쇼어(소인수 분해)와 그로버(탐색)가 있습니다. 현재는 노이즈로 규모가 작고 연구 중(NISQ). 응용 분야로는 화학 시뮬레이션, 최적화, 암호학 등이 있습니다.'''
print('글자 수 :', len(recorded_answer), '| 토큰 수 :', len(enc.encode(recorded_answer)))

글자 수 : 266 | 토큰 수 : 168


> **📝 고친 부분 (원본 cell 5 · 10)** 시스템 프롬프트의 '**100 토큰 이내로**' 는 부탁일 뿐이다 — 수업 결과는 위처럼 **100 토큰을 넘는다**. 모델은 자기가 쓴 토큰 수를 세지 못한다. 길이를 정말 묶으려면 매개변수(`ChatOpenAI(model='gpt-5-nano', max_completion_tokens=…)`, Day 046 참고)를 쓰고, 프롬프트에는 '**세 문장으로**' 처럼 사람이 확인할 수 있는 단위를 쓴다.

In [9]:
# 🧪 가짜 LLM : 정해 둔 메시지를 차례로 돌려준다 (API · 인터넷 · 요금 없음)
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

class ScriptedLLM(GenericFakeChatModel):
    """create_agent 도 쓸 수 있게 bind_tools 만 흉내 낸 가짜 모델"""
    def bind_tools(self, tools, **kwargs):
        return self

def fake_llm(*replies):
    return ScriptedLLM(messages=iter([r if isinstance(r, AIMessage) else AIMessage(content=r) for r in replies]))

In [10]:
# 🧪 모의 실행 — API 없이 정해 둔 답을 내는 가짜 LLM 으로 '흐름' 만 확인

fake_chain = prompt | fake_llm('양자 컴퓨터는 큐비트로 계산하는 컴퓨터입니다.') | output_parser
print(type(fake_chain).__name__, '— 단계 :', [type(s).__name__ for s in fake_chain.steps])
print(fake_chain.invoke({'topic': '양자 컴퓨터'}))

# 파서를 빼면 문자열이 아니라 메시지 객체(AIMessage)가 나온다 — 출력 파서가 하는 일
print(type((prompt | fake_llm('…')).invoke({'topic': 'x'})).__name__)

RunnableSequence — 단계 : ['ChatPromptTemplate', 'ScriptedLLM', 'StrOutputParser']
양자 컴퓨터는 큐비트로 계산하는 컴퓨터입니다.
AIMessage


- `prompt | llm | parser` 는 `RunnableSequence` 라는 **하나의 실행 객체**가 된다 — `invoke` 한 번에 세 단계가 차례로 돈다.
- 출력 파서를 빼면 `AIMessage` 객체가 나오고, 글은 `.content` 에 있다(Day 046 의 `choices[0].message.content` 와 같은 역할).

## 4. 고객 리뷰 감성 분석 체인 (필기 `LangChain_실습2`, 교재 p.24)

In [11]:
user_prompt = """
다음 한글로 작성된 <고객 리뷰>의 감성을 분석해서 판단 근거와 함께 결과를 긍정 또는 부정으로 제시하세요.
<고객 리뷰>
{review}
"""
sentiment_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 한글로 작성된 고객 리뷰의 감성을 분석하는 경력 10년 이상의 뛰어난 분석가 입니다.'),
    ('user', user_prompt),
])
reviews = ['배송도 빠르고 국물이 진해서 가족 모두 만족했어요. 재구매 의사 있습니다!',
           '고기가 너무 질기고 양도 적어요. 이 가격이면 다른 데서 사 먹겠습니다.',
           '맛은 괜찮은데 포장이 터져서 왔네요. 다음엔 꼼꼼히 포장해 주세요.']
print(sentiment_prompt.format_messages(review=reviews[0])[1].content)


다음 한글로 작성된 <고객 리뷰>의 감성을 분석해서 판단 근거와 함께 결과를 긍정 또는 부정으로 제시하세요.
<고객 리뷰>
배송도 빠르고 국물이 진해서 가족 모두 만족했어요. 재구매 의사 있습니다!



> **📝 고친 부분 (원본 `LangChain_실습2`)** 필기는 '**두 번째 구성 요소: llm 모델 생성**' 빈 셀에서 멈췄다 → LLM · 파서 · 체인 · 실행을 이어서 완성. 그리고 '판단 근거와 함께 결과를 긍정 또는 부정으로' 라고만 쓰면 답이 **자유로운 글**이라 코드로 결과를 모으기 어렵다(Day 046 감성 분석의 교훈) → 아래처럼 **구조화 출력**으로 형식을 고정한다.

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# (1) 필기 방식 그대로 — 결과는 자유 형식 글
llm = ChatOpenAI(model='gpt-5-nano')
sentiment_chain = sentiment_prompt | llm | StrOutputParser()
print(sentiment_chain.invoke({'review': reviews[0]}))

In [13]:
# (2) 결과 형식을 클래스로 정한다 — 모델이 이 칸들을 채운 객체를 돌려준다
from typing import Literal
from pydantic import BaseModel, Field

class Sentiment(BaseModel):
    label: Literal['긍정', '부정', '중립'] = Field(description='리뷰의 감성')
    reason: str = Field(description='판단 근거 (리뷰 속 표현을 인용해 한 문장)')
    keywords: list[str] = Field(description='판단에 쓴 핵심 표현 1~3개')

import json
print(json.dumps(Sentiment.model_json_schema()['properties'], ensure_ascii=False, indent=1))

{
 "label": {
  "description": "리뷰의 감성",
  "enum": [
   "긍정",
   "부정",
   "중립"
  ],
  "title": "Label",
  "type": "string"
 },
 "reason": {
  "description": "판단 근거 (리뷰 속 표현을 인용해 한 문장)",
  "title": "Reason",
  "type": "string"
 },
 "keywords": {
  "description": "판단에 쓴 핵심 표현 1~3개",
  "items": {
   "type": "string"
  },
  "title": "Keywords",
  "type": "array"
 }
}


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

structured_chain = sentiment_prompt | ChatOpenAI(model='gpt-5-nano').with_structured_output(Sentiment)
results = structured_chain.batch([{'review': r} for r in reviews])     # 여러 리뷰를 한 번에 (병렬 호출)
for r, s in zip(reviews, results):
    print(f'{s.label} | {s.reason} | {s.keywords} ← {r[:25]}…')

In [15]:
# 🧪 모의 실행 — API 없이 정해 둔 답을 내는 가짜 LLM 으로 '흐름' 만 확인

# 구조화 출력이 돌려주는 모양 — 가짜 JSON 답을 같은 클래스로 검사해 본다
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=Sentiment)
demo = (sentiment_prompt
        | fake_llm('{"label": "긍정", "reason": "\'국물이 진해서 가족 모두 만족\' 이라고 했다", "keywords": ["국물이 진해", "재구매"]}')
        | parser).invoke({'review': reviews[0]})
print(type(demo).__name__, '→', demo.label, '|', demo.reason, '|', demo.keywords)

try:                                               # 형식이 틀린 답이면 바로 오류로 잡힌다
    (sentiment_prompt | fake_llm('{"label": "좋음", "reason": "맛있다"}') | parser).invoke({'review': reviews[0]})
except Exception as e:
    print('형식 오류 감지 :', type(e).__name__)

Sentiment → 긍정 | '국물이 진해서 가족 모두 만족' 이라고 했다 | ['국물이 진해', '재구매']
형식 오류 감지 : OutputParserException


- `with_structured_output(클래스)` 를 쓰면 결과가 **글이 아니라 객체**로 와서 `s.label` 처럼 바로 쓸 수 있고, `'좋음'` 처럼 정해 두지 않은 값이나 빠진 칸은 오류로 잡힌다.
- 여러 리뷰는 `for` 대신 **`batch()`** — 내부에서 동시에 요청해 빠르다.

## 정리

| 구성 요소 | 클래스 | 하는 일 |
|---|---|---|
| 프롬프트 | `ChatPromptTemplate.from_messages([...])` | `{변수}` 를 채워 메시지 목록 생성 (`format_messages` 로 확인) |
| LLM | `ChatOpenAI(model='gpt-5-nano')` | 메시지 → `AIMessage` |
| 출력 파서 | `StrOutputParser()` · `PydanticOutputParser` | `AIMessage` → 글 · 객체 |
| 체인 | `prompt \| llm \| parser` | `invoke(dict)` · `batch([dict, …])` |
| 구조화 출력 | `llm.with_structured_output(클래스)` | 정해 둔 칸을 채운 객체 |